# Noise Response Overlay

This notebook generates a small direct voltage spectrogram from
white Gaussian voltage noise. It then overlays the measured
power bandpass with the ideal PFB response.

The run uses `digitize=False` and `requantize=False` so the
result isolates the PFB and fine-channelization behavior from
quantization effects.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pfb_response_tools import (
    PFBExperimentConfig,
    detected_intensity_sweep,
    ideal_response,
    local_noise_stats,
    modeled_bandpass_excess_sweep,
    modeled_bandpass_summary,
    noise_overlay_summary,
    normalize_column,
    run_spectrogram,
    tone_response_sweep,
)

plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})


In [ ]:
config = PFBExperimentConfig()
data = run_spectrogram(config, seed=20260502, noise=True)
measured = data.mean(axis=0)
measured_norm = measured / measured.mean()
response = ideal_response(config)

summary = noise_overlay_summary(config, seed=20260502)
summary


In [ ]:
fine = np.arange(config.fftlength)
fine_offset = (fine - config.fftlength / 2) * config.df / 1e3

fig, ax = plt.subplots()
ax.plot(fine_offset, measured_norm, lw=1.5, label="Measured noise mean")
ax.plot(fine_offset, response, lw=2, alpha=0.8, label="Ideal PFB response")
ax.set_xlabel("Fine-channel offset from coarse-channel center (kHz)")
ax.set_ylabel("Power / band mean")
ax.set_title("Noise bandpass vs ideal PFB response")
ax.legend()
display(fig)
plt.close(fig)


In [ ]:
measured_std = data.std(axis=0)
measured_std_norm = measured_std / measured_std.mean()

fig, ax = plt.subplots()
ax.plot(fine_offset, measured_norm, label="Mean power")
ax.plot(fine_offset, measured_std_norm, label="Std per pixel")
ax.plot(fine_offset, response, lw=2, alpha=0.75, label="Ideal response")
ax.set_xlabel("Fine-channel offset from coarse-channel center (kHz)")
ax.set_ylabel("Relative level")
ax.set_title("Noise mean and variance both inherit PFB structure")
ax.legend()
display(fig)
plt.close(fig)


In [ ]:
channel_index = config.fftlength // 2
noise_mean, noise_std, sample_count = local_noise_stats(
    data,
    channel_index,
    context_bins=32,
    guard_bins=3,
)
{
    "center channel index": channel_index,
    "local sigma-clipped mean": noise_mean,
    "local sigma-clipped std": noise_std,
    "samples after clipping": sample_count,
}


The overlay is the main sanity check: for white voltage noise,
measured power follows the PFB response. This supports our
recent decision to estimate SNR from local spectral windows:
local windows follow the nearby PFB structure, while a global
mean/std mixes coarse-channel positions with different response
levels.
